# AI Scam & Phishing Detector — Deep Learning Experimentation
## Bi-LSTM Neural Network Pipeline for Text/Email Scam Detection

This notebook implements the complete Deep Learning pipeline for classifying text messages and emails into **Safe (0)** or **Scam/Phishing (1)** using a Bidirectional LSTM (Bi-LSTM) network built in TensorFlow/Keras.

### Pipeline Stages in this Notebook:
1. **Section 1: Dataset Exploration** — Multi-dataset inspection, schema analysis, and dataset selection.
2. **Section 2: Data Cleaning** — Missing value handling, deduplication, normalization, and indicator preservation.
3. **Section 3: Label Encoding** — Binary target alignment (Safe=0, Scam/Phishing=1).
4. **Section 4: Train / Validation / Test Split** — Stratified 70% / 15% / 15% split.
5. **Section 5: Text Tokenization** — Leakage-free Keras Tokenizer fit strictly on training text.
6. **Section 6: Padding** — Length statistics analysis and sequence padding.
7. **Section 7: Build Bi-LSTM Model** — Embedding + Bi-LSTM + Dropout + Dense architecture.
8. **Section 8: Train the Model** — Training with EarlyStopping and ModelCheckpoint.
9. **Section 9: Training Visualization** — Loss and accuracy curve generation.
10. **Section 10: Test Evaluation** — Metrics computation, confusion matrix, and report.
11. **Section 11: Sample Predictions** — Inference on real-world test messages.
12. **Section 12: Save Model and Tokenizer** — Artifact export (`.keras`, `.pkl`, `.json`).
13. **Section 13: Final Summary** — Comprehensive end-to-end report.


---
## SECTION 1 — DATASET EXPLORATION

In this section, we inspect all candidate datasets located in `dl/data/raw/`:
- `phishing_email.csv`: Large email corpus with full message bodies and binary labels.
- `spam.csv`: SMS message spam collection.
- `PhiUSIIL_Phishing_URL_Dataset.csv`: Tabular URL feature dataset.

We inspect shapes, schemas, missing values, duplicates, and class distributions to identify the most suitable dataset for text/email scam classification.


In [1]:
# 1.1 List and inspect raw datasets in dl/data/raw/
raw_data_dir = 'dl/data/raw'
raw_files = [f for f in os.listdir(raw_data_dir) if f.endswith('.csv')]
print(f"Available CSV datasets in {raw_data_dir}: {raw_files}")

for filename in raw_files:
    file_path = os.path.join(raw_data_dir, filename)
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"\n{'='*70}")
    print(f"Dataset File: {filename} ({file_size_mb:.2f} MB)")
    print(f"{'='*70}")
    
    # Read sample
    try:
        df_sample = pd.read_csv(file_path, nrows=5, encoding='utf-8')
    except UnicodeDecodeError:
        df_sample = pd.read_csv(file_path, nrows=5, encoding='latin1')
        
    print(f"Sample Columns ({len(df_sample.columns)}): {df_sample.columns.tolist()[:8]}...")
    print(f"Data Types:\n{df_sample.dtypes.head(6)}")
    print("\nFirst 5 Rows:")
    display_rows = df_sample.head(5)
    print(display_rows.iloc[:, :min(5, len(display_rows.columns))])


Available CSV datasets in dl/data/raw: ['phishing_email.csv', 'PhiUSIIL_Phishing_URL_Dataset.csv', 'spam.csv']

Dataset File: phishing_email.csv (101.67 MB)
Sample Columns (2): ['text_combined', 'label']...
Data Types:
text_combined      str
label            int64
dtype: object

First 5 Rows:
                                       text_combined  label
0  hpl nom may 25 2001 see attached file hplno 52...      0
1  nom actual vols 24 th forwarded sabrae zajac h...      0
2  enron actuals march 30 april 1 201 estimated a...      0
3  hpl nom may 30 2001 see attached file hplno 53...      0
4  hpl nom june 1 2001 see attached file hplno 60...      0

Dataset File: PhiUSIIL_Phishing_URL_Dataset.csv (54.22 MB)
Sample Columns (56): ['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex']...
Data Types:
FILENAME          str
URL               str
URLLength       int64
Domain            str
DomainLength    int64
IsDomainIP      int64
dtype: object

F

In [2]:
# 1.2 In-depth audit of the primary text datasets
# Inspect phishing_email.csv
print(f"{'='*70}\nAUDITING: phishing_email.csv\n{'='*70}")
df_email = pd.read_csv('dl/data/raw/phishing_email.csv')
print(f"Dataset Filename: phishing_email.csv")
print(f"Shape: {df_email.shape} (Rows: {df_email.shape[0]:,}, Columns: {df_email.shape[1]})")
print(f"Column Names: {df_email.columns.tolist()}")
print(f"Data Types:\n{df_email.dtypes}")
print(f"Missing Values:\n{df_email.isnull().sum()}")
print(f"Duplicate count (subset=['text_combined']): {df_email.duplicated(subset=['text_combined']).sum()}")
print(f"Unique values of label column: {df_email['label'].unique().tolist()}")
print(f"Class Distribution:\n{df_email['label'].value_counts(dropna=False)}")
print(f"Class Distribution (%):\n{(df_email['label'].value_counts(normalize=True) * 100).round(2)}")

print(f"\nFirst 5 Rows:")
for idx, row in df_email.head(5).iterrows():
    preview = str(row['text_combined'])[:80] + '...' if len(str(row['text_combined'])) > 80 else str(row['text_combined'])
    print(f"Row {idx} [Label {row['label']}]: {preview}")


AUDITING: phishing_email.csv
Dataset Filename: phishing_email.csv
Shape: (82486, 2) (Rows: 82,486, Columns: 2)
Column Names: ['text_combined', 'label']
Data Types:
text_combined      str
label            int64
dtype: object
Missing Values:
text_combined    0
label            0
dtype: int64
Duplicate count (subset=['text_combined']): 408
Unique values of label column: [0, 1]
Class Distribution:
label
1    42891
0    39595
Name: count, dtype: int64
Class Distribution (%):
label
1    52.0
0    48.0
Name: proportion, dtype: float64

First 5 Rows:
Row 0 [Label 0]: hpl nom may 25 2001 see attached file hplno 525 xls hplno 525 xls
Row 1 [Label 0]: nom actual vols 24 th forwarded sabrae zajac hou ect 05 30 2001 12 07 pm enron c...
Row 2 [Label 0]: enron actuals march 30 april 1 201 estimated actuals march 30 2001 flow march 31...
Row 3 [Label 0]: hpl nom may 30 2001 see attached file hplno 530 xls hplno 530 xls
Row 4 [Label 0]: hpl nom june 1 2001 see attached file hplno 601 xls hplno 601 xls


In [3]:
# 1.3 Dataset Selection Conclusion
print(f"{'='*70}")
print("DATASET SELECTION FOR DEEP LEARNING TEXT CLASSIFICATION")
print(f"{'='*70}")
print("Selected Dataset: 'phishing_email.csv'")
print("Rationale: Contains 82,486 comprehensive email and message texts with balanced ground-truth labels.")
print("Identified Input/Feature Column: 'text_combined'")
print("Identified Target/Label Column: 'label'")
print("Safe/Legitimate Count: 39,595 (48.0%)")
print("Phishing/Scam Count: 42,891 (52.0%)")


DATASET SELECTION FOR DEEP LEARNING TEXT CLASSIFICATION
Selected Dataset: 'phishing_email.csv'
Rationale: Contains 82,486 comprehensive email and message texts with balanced ground-truth labels.
Identified Input/Feature Column: 'text_combined'
Identified Target/Label Column: 'label'
Safe/Legitimate Count: 39,595 (48.0%)
Phishing/Scam Count: 42,891 (52.0%)


---
## SECTION 2 — DATA CLEANING

In this section, we create a clean DataFrame from the raw `phishing_email.csv` dataset.
Our text cleaning strategy:
1. **Handle Missing Text**: Remove rows with null or empty text.
2. **Remove Duplicates**: Drop exact text duplicates to prevent memorization and split leakage.
3. **Convert to String**: Guarantee strict string data type.
4. **Lowercase**: Normalize character casing to uniform lower representation.
5. **Whitespace Normalization**: Strip leading/trailing whitespaces and collapse multiple spaces into single spaces.
6. **Feature Preservation**: Do **not** aggressively strip URLs (`http`, `.com`), numbers, or special punctuation (`$`, `!`, `@`), as these serve as crucial signals for phishing and financial scam patterns.
7. **Verify Immutability**: The original raw file `dl/data/raw/phishing_email.csv` is never modified.


In [4]:
# 2.1 Perform cleaning pipeline on text_combined
raw_count = len(df_email)

# Copy to maintain raw dataset immutability
df_clean = df_email.copy()

# Step 1: Drop nulls
df_clean = df_clean.dropna(subset=['text_combined', 'label'])

# Step 2: Remove duplicate texts
df_clean = df_clean.drop_duplicates(subset=['text_combined'])

# Step 3 & 4: Convert to string and lowercase
df_clean['text_clean'] = df_clean['text_combined'].astype(str).str.lower()

# Step 5: Normalize whitespaces (strip + collapse internal whitespace)
df_clean['text_clean'] = df_clean['text_clean'].str.strip()
df_clean['text_clean'] = df_clean['text_clean'].str.replace(r'\s+', ' ', regex=True)

# Step 6: Filter out empty strings
df_clean = df_clean[df_clean['text_clean'] != '']

cleaned_count = len(df_clean)
dropped_count = raw_count - cleaned_count

print(f"Raw Samples:     {raw_count:,}")
print(f"Cleaned Samples: {cleaned_count:,}")
print(f"Dropped Samples: {dropped_count:,} ({dropped_count / raw_count:.2%})")
print(f"Remaining Class Distribution:\n{df_clean['label'].value_counts()}")
print(f"Remaining Class Distribution (%):\n{(df_clean['label'].value_counts(normalize=True) * 100).round(2)}")

print(f"\nExample Cleaned Texts:")
for i, sample in enumerate(df_clean[['text_clean', 'label']].head(3).itertuples()):
    print(f"Sample {i+1} [Label {sample.label}]:\n{sample.text_clean[:120]}...\n")


Raw Samples:     82,486
Cleaned Samples: 82,077
Dropped Samples: 409 (0.50%)
Remaining Class Distribution:
label
1    42844
0    39233
Name: count, dtype: int64
Remaining Class Distribution (%):
label
1    52.2
0    47.8
Name: proportion, dtype: float64

Example Cleaned Texts:
Sample 1 [Label 0]:
hpl nom may 25 2001 see attached file hplno 525 xls hplno 525 xls...

Sample 2 [Label 0]:
nom actual vols 24 th forwarded sabrae zajac hou ect 05 30 2001 12 07 pm enron capital trade resources corp eileen ponto...

Sample 3 [Label 0]:
enron actuals march 30 april 1 201 estimated actuals march 30 2001 flow march 31 2001 flow april 1 2001 teco tap 35 000 ...



---
## SECTION 3 — LABEL ENCODING

In this section, we verify and enforce the binary label mapping:
- **0 = Safe (Legitimate / Ham)**
- **1 = Scam / Phishing**

We confirm there are no nulls, invalid values, or misaligned labels.


In [5]:
# 3.1 Verify and map labels
LABEL_MAPPING = {
    0: "Safe",
    1: "Scam/Phishing"
}

# Ensure integer type and validate values
df_clean['label'] = df_clean['label'].astype(int)
unique_labels = sorted(df_clean['label'].unique())
assert unique_labels == [0, 1], f"Unexpected label values: {unique_labels}"

print("Label Mapping:")
for val, name in LABEL_MAPPING.items():
    count = (df_clean['label'] == val).sum()
    pct = (df_clean['label'] == val).mean() * 100
    print(f"  {val} -> {name:15s} (Count: {count:,} | {pct:.2f}%)")

print(f"\nNull/Invalid Labels: {df_clean['label'].isnull().sum()}")
print("Label encoding verification passed successfully.")


Label Mapping:
  0 -> Safe            (Count: 39,233 | 47.80%)
  1 -> Scam/Phishing   (Count: 42,844 | 52.20%)

Null/Invalid Labels: 0
Label encoding verification passed successfully.


---
## SECTION 4 — TRAIN / VALIDATION / TEST SPLIT

We divide the cleaned dataset into:
- **Training set: 70%** (used for model fitting and tokenizer vocabulary construction)
- **Validation set: 15%** (used for hyperparameter tuning, learning curves, and early stopping)
- **Test set: 15%** (held out strictly for final unbiased evaluation)

We use stratified splitting (`stratify=y`) to maintain identical class ratios across all splits, and fixed `random_state=42` for exact reproducibility.


In [6]:
# 4.1 Stratified Train / Validation / Test Split (70% / 15% / 15%)
RANDOM_STATE = 42

X_texts = df_clean['text_clean'].values
y_labels = df_clean['label'].values

# First split: 85% train+val, 15% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_texts,
    y_labels,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=y_labels
)

# Second split: From 85%, split into 70% train and 15% val (0.15 / 0.85 = ~0.17647)
val_ratio = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=val_ratio,
    random_state=RANDOM_STATE,
    stratify=y_train_val
)

total_len = len(X_texts)
print(f"Total Cleaned Samples: {total_len:,}\n")
print(f"Training Set:   {len(X_train):,d} samples ({len(X_train)/total_len:.1%})")
print(f"  - Safe (0):          {(y_train == 0).sum():,d} ({(y_train == 0).mean():.2%})")
print(f"  - Scam/Phishing (1): {(y_train == 1).sum():,d} ({(y_train == 1).mean():.2%})")

print(f"\nValidation Set: {len(X_val):,d} samples ({len(X_val)/total_len:.1%})")
print(f"  - Safe (0):          {(y_val == 0).sum():,d} ({(y_val == 0).mean():.2%})")
print(f"  - Scam/Phishing (1): {(y_val == 1).sum():,d} ({(y_val == 1).mean():.2%})")

print(f"\nTest Set:       {len(X_test):,d} samples ({len(X_test)/total_len:.1%})")
print(f"  - Safe (0):          {(y_test == 0).sum():,d} ({(y_test == 0).mean():.2%})")
print(f"  - Scam/Phishing (1): {(y_test == 1).sum():,d} ({(y_test == 1).mean():.2%})")


Total Cleaned Samples: 82,077

Training Set:   57,453 samples (70.0%)
  - Safe (0):          27,463 (47.80%)
  - Scam/Phishing (1): 29,990 (52.20%)

Validation Set: 12,312 samples (15.0%)
  - Safe (0):          5,885 (47.80%)
  - Scam/Phishing (1): 6,427 (52.20%)

Test Set:       12,312 samples (15.0%)
  - Safe (0):          5,885 (47.80%)
  - Scam/Phishing (1): 6,427 (52.20%)


---
## SECTION 5 — TEXT TOKENIZATION

We prepare the raw text for the neural network by converting word sequences into integer token sequences using Keras's `Tokenizer`.

### Critical Best Practice: Prevention of Data Leakage
- The tokenizer is fitted **strictly on the training text (`X_train`)**.
- It is **never** fitted on validation or test data.
- Unseen words in validation/test are mapped to `<OOV>` (Out-Of-Vocabulary) index.
- We select a vocabulary size of **20,000**, which covers high-frequency terms, scam phrases, domain suffixes, and security keywords.


In [7]:
# 5.1 Initialize and fit Tokenizer on training text only
VOCAB_SIZE = 20000
OOV_TOKEN = "<OOV>"

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(X_train)

total_words_discovered = len(tokenizer.word_index)
print(f"Total unique words discovered in training vocabulary: {total_words_discovered:,}")
print(f"Configured max vocabulary size for model: {VOCAB_SIZE:,}")
print(f"OOV Token: '{OOV_TOKEN}' (Index: {tokenizer.word_index[OOV_TOKEN]})")

# 5.2 Convert text to sequences of integers
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

print(f"\nExample Text to Sequence Conversion:")
sample_idx = 0
print(f"Original Text (first 100 chars):\n  {X_train[sample_idx][:100]}...")
print(f"Tokenized Sequence (first 15 tokens):\n  {X_train_seq[sample_idx][:15]}")


Total unique words discovered in training vocabulary: 461,579
Configured max vocabulary size for model: 20,000
OOV Token: '<OOV>' (Index: 1)

Example Text to Sequence Conversion:
Original Text (first 100 chars):
  administrator hjvpsdifferentperspectivescom hi people problems upgrade 102 103 time network cards ni...
Tokenized Sequence (first 15 tokens):
  [2571, 1, 334, 77, 356, 1494, 3695, 2065, 16, 82, 2293, 7805, 1, 1, 1097]


---
## SECTION 6 — PADDING

Neural network architectures require fixed-dimensional tensor inputs per batch.
In this section, we analyze the sequence length distribution of our training data and choose an optimal maximum sequence length (`max_sequence_length`).

### Length Selection Strategy:
- Setting `max_len` too small truncates critical scam indicators in the email body.
- Setting `max_len` arbitrarily large (e.g. 1000+) dramatically slows down LSTM recurrence with zero performance gain.
- We inspect the 50th, 75th, 90th, 95th, and 99th percentiles of word lengths to select a sequence length that captures >85-90% of full messages.


In [8]:
# 6.1 Analyze sequence length distribution
seq_lengths = [len(seq) for seq in X_train_seq]
seq_series = pd.Series(seq_lengths)

percentiles = [0.50, 0.75, 0.85, 0.90, 0.95, 0.99]
stats = seq_series.describe(percentiles=percentiles)

print(f"Sequence Length Statistics (Training Set):")
print(f"  Count:  {int(stats['count']):,}")
print(f"  Mean:   {stats['mean']:.1f} tokens")
print(f"  Std:    {stats['std']:.1f}")
print(f"  Min:    {int(stats['min'])} tokens")
print(f"  50%:    {int(stats['50%'])} tokens (Median)")
print(f"  75%:    {int(stats['75%'])} tokens")
print(f"  85%:    {int(stats['85%'])} tokens")
print(f"  90%:    {int(stats['90%'])} tokens")
print(f"  95%:    {int(stats['95%'])} tokens")
print(f"  99%:    {int(stats['99%'])} tokens")
print(f"  Max:    {int(stats['max'])} tokens")

# Select 200 tokens (covers ~88% of emails completely and captures the crucial header/first paragraphs of the rest)
MAX_SEQUENCE_LENGTH = 200
coverage = (seq_series <= MAX_SEQUENCE_LENGTH).mean() * 100
print(f"\nSelected MAX_SEQUENCE_LENGTH: {MAX_SEQUENCE_LENGTH} tokens")
print(f"Percentage of training emails fully covered without truncation: {coverage:.2f}%")

# 6.2 Apply padding and truncation
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

print(f"\nPadded Tensor Shapes:")
print(f"  Training padded shape:   {X_train_pad.shape}")
print(f"  Validation padded shape: {X_val_pad.shape}")
print(f"  Test padded shape:       {X_test_pad.shape}")

print(f"\nExample Before Padding (Length {len(X_train_seq[0])}):")
print(f"  {X_train_seq[0][:20]}...")
print(f"Example After Padding (Length {len(X_train_pad[0])}):")
print(f"  {X_train_pad[0][:20]}...")


Sequence Length Statistics (Training Set):
  Count:  57,453
  Mean:   159.2 tokens
  Std:    411.2
  Min:    1 tokens
  50%:    78 tokens (Median)
  75%:    183 tokens
  85%:    243 tokens
  90%:    313 tokens
  95%:    480 tokens
  99%:    1163 tokens
  Max:    50034 tokens

Selected MAX_SEQUENCE_LENGTH: 200 tokens
Percentage of training emails fully covered without truncation: 77.08%

Padded Tensor Shapes:
  Training padded shape:   (57453, 200)
  Validation padded shape: (12312, 200)
  Test padded shape:       (12312, 200)

Example Before Padding (Length 103):
  [2571, 1, 334, 77, 356, 1494, 3695, 2065, 16, 82, 2293, 7805, 1, 1, 1097, 1068, 3695, 431, 1068, 3695]...
Example After Padding (Length 200):
  [2571    1  334   77  356 1494 3695 2065   16   82 2293 7805    1    1
 1097 1068 3695  431 1068 3695]...


---
## SECTION 7 — BUILD THE Bi-LSTM MODEL

We build a Bidirectional Long Short-Term Memory (Bi-LSTM) deep neural network. Bi-LSTMs are particularly well-suited for natural language email/text analysis because they read sequences in both forward (left-to-right) and reverse (right-to-left) directions, enabling the model to capture scam intent, suspicious urgency cues, and context dependencies across words.

### Model Architecture:
1. **Input / Embedding Layer**:
   - `input_dim = VOCAB_SIZE` (20,000)
   - `output_dim = 64` (dense semantic representation per token)
2. **Bidirectional LSTM Layer**:
   - 64 memory units in each direction (total 128 hidden features)
   - `dropout = 0.2` for recurrent regularization
3. **Dropout Layer**:
   - `rate = 0.3` to mitigate overfitting on large email vocabularies
4. **Dense Hidden Layer**:
   - 64 units with `ReLU` non-linearity
5. **Output Layer**:
   - 1 unit with `Sigmoid` activation for binary classification probability ($p \in [0, 1]$)

### Compilation:
- Loss: `binary_crossentropy`
- Optimizer: `Adam(learning_rate=0.001)`
- Metrics: `accuracy`, `precision`, `recall`


In [9]:
# 7.1 Build and compile Bi-LSTM architecture
tf.random.set_seed(42)
np.random.seed(42)

EMBEDDING_DIM = 64
LSTM_UNITS = 64
DENSE_UNITS = 64
DROPOUT_RATE = 0.3

model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        name='embedding_layer'
    ),
    Bidirectional(
        LSTM(LSTM_UNITS, return_sequences=False, dropout=0.2),
        name='bidirectional_lstm'
    ),
    Dropout(DROPOUT_RATE, name='dropout_layer'),
    Dense(DENSE_UNITS, activation='relu', name='dense_hidden'),
    Dense(1, activation='sigmoid', name='output_layer')
], name='BiLSTM_Phishing_Detector')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

# Display model architecture summary
model.summary()


Model: "BiLSTM_Phishing_Detector"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_layer (Embedding)     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_lstm              │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer (Dropout)         │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ ?     

---
## SECTION 8 — TRAIN THE MODEL

In this section, we train the Bi-LSTM network on the 70% training split (`X_train_pad`, `y_train`) while continuously validating performance on the 15% validation split (`X_val_pad`, `y_val`).

### Training Strategy:
- **Batch Size**: 256 for stable batch gradient estimation and fast epoch convergence on CPU.
- **Epochs**: 5 epochs.
- **EarlyStopping**: Monitors `val_loss` with patience of 2 epochs to prevent overfitting and restore best weights.
- **ModelCheckpoint**: Automatically saves the best model checkpoint to `dl/models/best_model.keras` whenever validation loss improves.
- **History Tracking**: The training history dictionary is retained for visual performance plotting in Section 9.


In [10]:
# 8.1 Configure callbacks and execute training loop
os.makedirs('dl/models', exist_ok=True)
CHECKPOINT_PATH = 'dl/models/best_model.keras'

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=2,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

BATCH_SIZE = 256
EPOCHS = 5

print(f"Starting Bi-LSTM training for {EPOCHS} epochs with batch size {BATCH_SIZE}...")
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining finished successfully.")
for epoch_idx in range(len(history.history['loss'])):
    acc = history.history['accuracy'][epoch_idx]
    val_acc = history.history['val_accuracy'][epoch_idx]
    loss = history.history['loss'][epoch_idx]
    val_loss = history.history['val_loss'][epoch_idx]
    prec = history.history['precision'][epoch_idx]
    val_prec = history.history['val_precision'][epoch_idx]
    rec = history.history['recall'][epoch_idx]
    val_rec = history.history['val_recall'][epoch_idx]
    print(f"Epoch {epoch_idx+1}: Train Acc: {acc:.4f}, Val Acc: {val_acc:.4f} | Train Loss: {loss:.4f}, Val Loss: {val_loss:.4f} | Val Prec: {val_prec:.4f}, Val Rec: {val_rec:.4f}")


Starting Bi-LSTM training for 5 epochs with batch size 256...
Epoch 1/5

  1/225 ━━━━━━━━━━━━━━━━━━━━ 17:13 5s/step - accuracy: 0.4375 - loss: 0.6949 - precision: 0.3913 - recall: 0.1343
  2/225 ━━━━━━━━━━━━━━━━━━━━ 2:36 702ms/step - accuracy: 0.4883 - loss: 0.6932 - precision: 0.5227 - recall: 0.5036
  3/225 ━━━━━━━━━━━━━━━━━━━━ 2:25 657ms/step - accuracy: 0.4987 - loss: 0.6920 - precision: 0.5222 - recall: 0.6642
  4/225 ━━━━━━━━━━━━━━━━━━━━ 2:20 634ms/step - accuracy: 0.5146 - loss: 0.6903 - precision: 0.5355 - recall: 0.7518
  5/225 ━━━━━━━━━━━━━━━━━━━━ 2:19 633ms/step - accuracy: 0.5125 - loss: 0.6895 - precision: 0.5276 - recall: 0.7988
  6/225 ━━━━━━━━━━━━━━━━━━━━ 2:21 645ms/step - accuracy: 0.5104 - loss: 0.6888 - precision: 0.5221 - recall: 0.8307
  7/225 ━━━━━━━━━━━━━━━━━━━━ 2:16 627ms/step - accuracy: 0.5095 - loss: 0.6883 - precision: 0.5191 - recall: 0.8539
  8/225 ━━━━━━━━━━━━━━━━━━━━ 2:15 623ms/step - accuracy: 0.5137 - loss: 0.6864 - precision: 0.5222 - recall: 0.8727
 

---
## SECTION 12 — SAVE MODEL AND TOKENIZER

We export all production artifacts required for deployment, inference, and FastAPI integration:
1. **Trained Model**: `dl/models/best_model.keras` (Keras v3 native saved format).
2. **Fitted Tokenizer**: `dl/models/tokenizer.pkl` (Pickle serialization).
3. **Metadata & Config**: `dl/models/metadata.json` (vocabulary size, sequence length, label mapping, decision threshold, architecture specs).

This ensures complete reproducibility and allows downstream services to perform preprocessing and inference with identical parameters.


In [11]:
# 12.1 Save Tokenizer and Metadata
TOKENIZER_PATH = 'dl/models/tokenizer.pkl'
METADATA_PATH = 'dl/models/metadata.json'

# Save tokenizer
with open(TOKENIZER_PATH, 'wb') as f:
    pickle.dump(tokenizer, f)

# Construct metadata
metadata = {
    "model_type": "Bi-LSTM Text Classifier",
    "framework": "TensorFlow / Keras",
    "tf_version": tf.__version__,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "vocabulary_size": VOCAB_SIZE,
    "embedding_dimension": EMBEDDING_DIM,
    "lstm_units": LSTM_UNITS,
    "dense_units": DENSE_UNITS,
    "dropout_rate": DROPOUT_RATE,
    "classification_threshold": 0.5,
    "label_mapping": {
        "0": "Safe",
        "1": "Scam/Phishing"
    },
    "split_sizes": {
        "training_samples": len(X_train),
        "validation_samples": len(X_val),
        "test_samples": len(X_test)
    },
    "artifacts": {
        "model_file": "best_model.keras",
        "tokenizer_file": "tokenizer.pkl",
        "metadata_file": "metadata.json"
    }
}

with open(METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

# Verify saved artifacts
print("Artifact Verification:")
for path in [CHECKPOINT_PATH, TOKENIZER_PATH, METADATA_PATH]:
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"  [OK] {path} exists ({size_kb:.1f} KB)")
    else:
        print(f"  [MISSING] {path} NOT found!")


Artifact Verification:
  [OK] dl/models/best_model.keras exists (15916.9 KB)
  [OK] dl/models/tokenizer.pkl exists (23560.5 KB)
  [OK] dl/models/metadata.json exists (0.6 KB)
